# Train trên Google Colab (GPU miễn phí)

Hướng dẫn nhanh:
1. Mở https://colab.research.google.com → **File → Upload notebook** → tải file này lên.
2. **Runtime → Change runtime type → T4 GPU**.
3. Chạy lần lượt các cell.

Có 2 cách nạp dữ liệu — chọn 1 trong cell 3:
- **Cách A (khuyến nghị):** tải dataset trực tiếp từ Kaggle bằng API token.
- **Cách B:** nén thư mục `Data` của bạn thành `Data.zip`, upload lên Google Drive rồi giải nén.

In [ ]:
# 1) Kiểm tra GPU
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 2) Lấy code từ GitHub
GITHUB_REPO_URL = 'https://github.com/<user>/<repo>.git'  # <-- THAY URL repo của bạn
!git clone $GITHUB_REPO_URL project
%cd project
!pip install -q imagehash grad-cam

In [ ]:
# 3) CÁCH A: tải dataset từ Kaggle
#    - Vào Kaggle → Account → Create New API Token để tải kaggle.json
#    - Chạy cell, khi hiện nút upload thì chọn file kaggle.json
from google.colab import files
files.upload()  # chọn kaggle.json

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d kelixo25/31-classes-of-skin-disease -p /content/data
!cd /content/data && unzip -q -o '*.zip'

# Tìm thư mục chứa train/ và test/
import glob
for p in glob.glob('/content/data/**/train', recursive=True):
    print('train tìm thấy tại:', p)

In [ ]:
# 3-bis) CÁCH B (thay cho Cách A): dùng Data.zip trên Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -q -o /content/drive/MyDrive/Data.zip -d /content/data

In [ ]:
# 4) Trỏ biến môi trường tới dataset và output.
#    SKIN_DATA_ROOT phải là thư mục chứa 'train' và 'test'.
import os
os.environ['SKIN_DATA_ROOT'] = '/content/data'   # <-- sửa cho khớp cell trên
os.environ['SKIN_OUTPUT_ROOT'] = '/content/project/outputs'

import glob
print('train:', len(glob.glob(os.environ['SKIN_DATA_ROOT'] + '/train/*')), 'lớp')
print('test :', len(glob.glob(os.environ['SKIN_DATA_ROOT'] + '/test/*')), 'lớp')

In [ ]:
# 5) Chuẩn bị dữ liệu
!python -m src.data.explore_dataset
!python -m src.data.validate_images
!python -m src.data.detect_duplicates
!python -m src.data.create_validation_split

In [ ]:
# 6) Train toàn bộ 4 mô hình
!python -m src.training.train --all-models

In [ ]:
# 7) Đánh giá + Grad-CAM + báo cáo
!python -m src.evaluation.evaluate --all-models
!python -m src.explainability.generate_gradcam_samples --all-models
!python -m src.evaluation.generate_summary

In [ ]:
# 8) Xem kết quả và lưu về Google Drive để không mất khi hết phiên
import pandas as pd
print(pd.read_csv('/content/project/outputs/reports/model_comparison.csv'))

from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/project/outputs /content/drive/MyDrive/skin_disease_outputs
print('Đã sao kết quả vào Google Drive: MyDrive/skin_disease_outputs')